# 🤖 RetailMind: Demand Forecasting & RFM Clustering

Demonstrates pipeline construction, feature transformation via ColumnTransformer, Random Forest regression training, and K-Means customer segmentation.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score, mean_absolute_error, silhouette_score
import sys
sys.path.append('..')
from src.preprocess import prepare_sales_features

## 1. Regression Pipeline Construction

In [ ]:
sales_df = pd.read_csv('../data/retail_sales.csv')
X, y = prepare_sales_features(sales_df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

cat_features = ['category']
num_features = ['store_id', 'price', 'discount', 'is_holiday', 'is_weekend', 'advertising_spend', 'customer_footfall', 'month']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
        ('num', StandardScaler(), num_features)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=120, max_depth=12, random_state=42))
])
pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
print(f'Test R² Score: {r2_score(y_test, y_pred)*100:.2f}%')
print(f'Test MAE:     {mean_absolute_error(y_test, y_pred):.2f} units')

## 2. Customer RFM Clustering

In [ ]:
rfm_df = pd.read_csv('../data/customer_rfm.csv')
rfm_X = rfm_df[['recency_days', 'frequency', 'monetary_spend']]

rfm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=4, random_state=42, n_init=10))
])
rfm_pipeline.fit(rfm_X)

scaled_X = rfm_pipeline.named_steps['scaler'].transform(rfm_X)
sil = silhouette_score(scaled_X, rfm_pipeline.named_steps['kmeans'].labels_)
print(f'Silhouette Score: {sil:.3f}')